In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os
from tqdm import tqdm
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint,ReduceLROnPlateau

In [ ]:
DATA_ROOT = os.environ.get("HIEROGLYPHS_DATA_DIR", "data/egyptian-hieroglyphs")
train_path=os.path.join(DATA_ROOT, "train")
valid_path=os.path.join(DATA_ROOT, "valid")
test_path=os.path.join(DATA_ROOT, "test")

train_annotation_path=os.path.join(train_path, "_annotations.csv")
valid_annotation_path=os.path.join(valid_path, "_annotations.csv")
test_annotation_path=os.path.join(test_path, "_annotations.csv")

In [ ]:
img_size = 224
train_classes_encoder = OneHotEncoder(sparse=False)
valid_classes_encoder = OneHotEncoder(sparse=False)
test_classes_encoder = OneHotEncoder(sparse=False)

In [ ]:
train_df = pd.read_csv(train_annotation_path)
train_df

In [ ]:
x_train = []
y_train = []
for i in tqdm(range(len(train_df))):
    img = cv2.imread(f"{train_path}/{train_df['filename'][i]}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img[train_df['ymin'][i]:train_df['ymax'][i], train_df['xmin'][i]:train_df['xmax'][i]]
    img = cv2.resize(img, (img_size, img_size))
    x_train.append(img)
    y_train.append(train_df['class'][i])

In [ ]:
plt.figure(figsize=(20,20))
for n,i in enumerate(np.random.randint(0,len(train_df),16)):
    plt.subplot(4,4,n+1)
    plt.imshow(x_train[i])
    plt.title(y_train[i])
    plt.axis('off')
plt.show()

In [ ]:
valid_df = pd.read_csv(valid_annotation_path)
valid_df

In [ ]:
x_valid = []
y_valid = []
for i in tqdm(range(len(valid_df))):
    img = cv2.imread(f"{valid_path}/{valid_df['filename'][i]}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img[valid_df['ymin'][i]:valid_df['ymax'][i], valid_df['xmin'][i]:valid_df['xmax'][i]]
    img = cv2.resize(img, (img_size, img_size))
    x_valid.append(img)
    y_valid.append(valid_df['class'][i])

In [ ]:
plt.figure(figsize=(20,20))
for n,i in enumerate(np.random.randint(0,len(valid_df),16)):
    plt.subplot(4,4,n+1)
    plt.imshow(x_valid[i])
    plt.title(y_valid[i])
    plt.axis('off')
plt.show()

In [ ]:
test_df = pd.read_csv(test_annotation_path)
test_df

In [ ]:
x_test = []
y_test = []
for i in tqdm(range(len(test_df))):
    img = cv2.imread(f"{test_path}/{test_df['filename'][i]}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img[test_df['ymin'][i]:test_df['ymax'][i], test_df['xmin'][i]:test_df['xmax'][i]]
    img = cv2.resize(img, (img_size, img_size))
    x_test.append(img)
    y_test.append(test_df['class'][i])

In [ ]:
plt.figure(figsize=(20,20))
for n,i in enumerate(np.random.randint(0,len(test_df),16)):
    plt.subplot(4,4,n+1)
    plt.imshow(x_test[i])
    plt.title(y_test[i])
    plt.axis('off')
plt.show()

In [ ]:
y_train = train_classes_encoder.fit_transform(np.array(y_train).reshape(-1,1))

y_valid = valid_classes_encoder.fit_transform(np.array(y_valid).reshape(-1,1))

y_test = test_classes_encoder.fit_transform(np.array(y_test).reshape(-1,1))

In [ ]:
x_train = np.array(x_train)
x_valid = np.array(x_valid)
x_test = np.array(x_test)

In [ ]:
models = { 
"Xception": tf.keras.applications.Xception,

"VGG16": tf.keras.applications.VGG16,

"VGG19": tf.keras.applications.VGG19,

"ResNet50": tf.keras.applications.ResNet50,

"ResNet101": tf.keras.applications.ResNet101,

"ResNet152": tf.keras.applications.ResNet152,

"InceptionV3": tf.keras.applications.InceptionV3,

"MobileNet": tf.keras.applications.MobileNet,
    
"MobileNetV2": tf.keras.applications.MobileNetV2,

"DenseNet121": tf.keras.applications.DenseNet121,

"DenseNet169": tf.keras.applications.DenseNet169,

"DenseNet201": tf.keras.applications.DenseNet201,

"NASNetMobile": tf.keras.applications.NASNetMobile,

"NASNetLarge": tf.keras.applications.NASNetLarge,

"EfficientNetB0": tf.keras.applications.EfficientNetB0,

"EfficientNetB1": tf.keras.applications.EfficientNetB1,

"EfficientNetB2": tf.keras.applications.EfficientNetB2,

"EfficientNetB3": tf.keras.applications.EfficientNetB3,

"EfficientNetB4": tf.keras.applications.EfficientNetB4,
    
"EfficientNetB5": tf.keras.applications.EfficientNetB5,

"EfficientNetB6": tf.keras.applications.EfficientNetB6,

"EfficientNetB7": tf.keras.applications.EfficientNetB7
}

In [ ]:
input_shape = (224, 224, 3)  

inputs = tf.keras.Input(shape=input_shape)
print(inputs )
model_name='ResNet152'
base_model = models[model_name](include_top=False, weights='imagenet', input_tensor=inputs)
base_model.trainable = False

x = tf.keras.layers.Flatten()(base_model.output)
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(95, activation='softmax')(x)

model = tf.keras.models.Model(inputs = inputs, outputs = outputs)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)

model.compile(loss="categorical_crossentropy", optimizer= optimizer, metrics=["accuracy",'precision','recall'])

model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

model_ReduceLROnPlateau = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=15, min_lr=0.000001)

callbacks=[early_stopping,model_checkpoint,model_ReduceLROnPlateau]

In [ ]:
history = model.fit(
                x_train,
                y_train,
                epochs = 20, 
                validation_data = (x_valid, y_valid),
                batch_size = 16,
                callbacks = callbacks,
                shuffle = True)

In [ ]:
import seaborn as sns
sns.set()

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall =history.history['recall']
val_recall =history.history['val_recall']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(loss) + 1)

In [ ]:
plt.plot(epochs, acc, color='green', label='Training Accuracy')
plt.plot(epochs, val_acc, color='blue', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.ylim(0,1.02)
plt.show()

In [ ]:
plt.figure()
plt.plot(epochs, precision, color='green', label='Training precision')
plt.plot(epochs, val_precision, color='red', label='Validation precision')
plt.title('Training and Validation precision')
plt.xlabel('Epoch')
plt.ylabel('precision')
plt.legend()
plt.ylim(0,1.02)
plt.show()

In [ ]:
plt.figure()
plt.plot(epochs, recall, color='green', label='Training Recall')
plt.plot(epochs, val_recall, color='red', label='Validation Recall')
plt.title('Training and Validation Recall')
plt.xlabel('Epoch')
plt.ylabel('Recall')
plt.legend()
plt.ylim(0,1.02)
plt.show()

In [ ]:
plt.figure()
plt.plot(epochs, loss, color='green', label='Training Loss')
plt.plot(epochs, val_loss, color='red', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.ylim(0,0.2)
plt.show()

In [ ]:
model_evaluate = model.evaluate(x_train,y_train)
print("Loss     : ",model_evaluate[0])
print("Accuracy : ",model_evaluate[1])
print("Precision  : ",model_evaluate[2])
print("Recall : ",model_evaluate[3])

In [ ]:
model_evaluate = model.evaluate(x_valid,y_valid)
print("Loss     : ",model_evaluate[0])
print("Accuracy : ",model_evaluate[1])
print("Precision  : ",model_evaluate[2])
print("Recall : ",model_evaluate[3])

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score,accuracy_score

y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

y_train_pred_class = np.argmax(y_train_pred, axis=1)
y_test_pred_class = np.argmax(y_test_pred, axis=1)

y_train_true = np.argmax(y_train, axis=1)
y_test_true = np.argmax(y_test, axis=1)

accuracy_train = accuracy_score(y_train_true, y_train_pred_class)
accuracy_test = accuracy_score(y_test_true, y_test_pred_class)

f1_train = f1_score(y_train_true, y_train_pred_class, average='weighted')
f1_test = f1_score(y_test_true, y_test_pred_class, average='weighted')

precision_train = precision_score(y_train_true, y_train_pred_class, average='weighted')
precision_test = precision_score(y_test_true, y_test_pred_class, average='weighted')

recall_train = recall_score(y_train_true, y_train_pred_class, average='weighted')
recall_test = recall_score(y_test_true, y_test_pred_class, average='weighted')

print(f"Training Accuracy: {accuracy_train:.2f}")
print(f"Test Accuracy: {accuracy_test:.2f}")
print(f"Training F1 Score: {f1_train:.2f}")
print(f"Test F1 Score: {f1_test:.2f}")
print(f"Training Precision: {precision_train:.2f}")
print(f"Test Precision: {precision_test:.2f}")
print(f"Training Recall: {recall_train:.2f}")
print(f"Test Recall: {recall_test:.2f}")

In [ ]:
data = {
    'model': ['TL(ResNet152)'],
    'Accuracy train': [accuracy_train],
    'Accuracy test': [accuracy_test],
    'Precision train': [precision_train],
    'Precision test': [precision_test],
    'Recall train': [recall_train],
    'Recall test': [recall_test],
    'F1-score train': [f1_train],
    'F1-score test': [f1_test]
}


metrics_df = pd.DataFrame(data)
metrics_df